# PediLens Wound Detection Model Training

This notebook trains a U-Net model for wound boundary detection and converts it to CoreML format.

**Steps:**
1. Setup environment
2. Upload or download dataset
3. Train model
4. Convert to CoreML
5. Download model for use in PediLens app

**Runtime:** GPU (Runtime > Change runtime type > GPU)

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q tensorflow coremltools Pillow opencv-python matplotlib seaborn

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import coremltools as ct
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
import json
from pathlib import Path

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ CoreML Tools version: {ct.__version__}")
print(f"✅ GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Step 2: Upload Dataset

**Option A:** Upload your own dataset (images and masks)

**Option B:** Download from Kaggle (requires API key)

**Expected structure:**
```
wound_data/
  images/
    image001.jpg
    image002.jpg
  masks/
    image001.png
    image002.png
```

In [ ]:
# Option A: Upload dataset from your computer
from google.colab import files

print("Upload your dataset as a ZIP file containing 'images' and 'masks' folders")
uploaded = files.upload()

# Extract ZIP
import zipfile
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('wound_data')
        print(f"✅ Extracted {filename}")

# Verify structure
!ls -la wound_data/

In [ ]:
# Option B: Download from Kaggle (uncomment if using)
# !pip install -q kaggle
# from google.colab import files
# 
# print("Upload your kaggle.json file")
# uploaded = files.upload()
# 
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# 
# # Download dataset (replace with actual dataset name)
# !kaggle datasets download -d <username>/<dataset-name>
# !unzip <dataset-name>.zip -d wound_data/

## Step 3: Configuration

In [ ]:
# Training configuration
CONFIG = {
    'image_size': (512, 512),
    'batch_size': 8,
    'epochs': 50,
    'learning_rate': 0.001,
    'validation_split': 0.2,
    'data_dir': './wound_data',
    'output_dir': './models',
    'model_name': 'WoundSegmentation'
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Step 4: Data Loading and Preprocessing

In [ ]:
def load_dataset(data_dir, image_size):
    """Load wound images and segmentation masks"""
    images_dir = Path(data_dir) / 'images'
    masks_dir = Path(data_dir) / 'masks'
    
    image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))
    
    images = []
    masks = []
    
    print(f"Loading {len(image_files)} images...")
    
    for img_path in image_files:
        # Load image
        img = Image.open(img_path).convert('RGB')
        img = img.resize(image_size)
        img_array = np.array(img) / 255.0
        images.append(img_array)
        
        # Load mask
        mask_path = masks_dir / img_path.name
        if mask_path.exists():
            mask = Image.open(mask_path).convert('L')
            mask = mask.resize(image_size)
            mask_array = np.array(mask) / 255.0
            mask_array = np.expand_dims(mask_array, axis=-1)
            masks.append(mask_array)
        else:
            print(f"⚠️ Mask not found for {img_path.name}")
            masks.append(np.zeros((*image_size, 1)))
    
    return np.array(images), np.array(masks)

def augment_data(images, masks):
    """Apply data augmentation"""
    augmented_images = []
    augmented_masks = []
    
    for img, mask in zip(images, masks):
        # Original
        augmented_images.append(img)
        augmented_masks.append(mask)
        
        # Horizontal flip
        augmented_images.append(np.fliplr(img))
        augmented_masks.append(np.fliplr(mask))
        
        # Vertical flip
        augmented_images.append(np.flipud(img))
        augmented_masks.append(np.flipud(mask))
        
        # Rotation 90
        augmented_images.append(np.rot90(img))
        augmented_masks.append(np.rot90(mask))
    
    return np.array(augmented_images), np.array(augmented_masks)

# Load and augment data
images, masks = load_dataset(CONFIG['data_dir'], CONFIG['image_size'])
print(f"✅ Loaded {len(images)} images")

images, masks = augment_data(images, masks)
print(f"✅ After augmentation: {len(images)} images")

# Split data
split_idx = int(len(images) * (1 - CONFIG['validation_split']))
train_images, val_images = images[:split_idx], images[split_idx:]
train_masks, val_masks = masks[:split_idx], masks[split_idx:]

print(f"✅ Training set: {len(train_images)} images")
print(f"✅ Validation set: {len(val_images)} images")

In [ ]:
# Visualize sample data
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(3):
    axes[0, i].imshow(train_images[i])
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(train_masks[i].squeeze(), cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Step 5: Build U-Net Model

In [ ]:
def create_unet_model(input_shape=(512, 512, 3), num_classes=1):
    """Create U-Net architecture for segmentation"""
    inputs = keras.Input(shape=input_shape)
    
    # Encoder
    conv1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)
    
    conv2 = layers.Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)
    
    conv3 = layers.Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)
    
    conv4 = layers.Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = layers.Conv2D(512, 3, activation='relu', padding='same')(conv4)
    drop4 = layers.Dropout(0.5)(conv4)
    pool4 = layers.MaxPooling2D(pool_size=(2, 2))(drop4)
    
    # Bottleneck
    conv5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(pool4)
    conv5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(conv5)
    drop5 = layers.Dropout(0.5)(conv5)
    
    # Decoder
    up6 = layers.Conv2D(512, 2, activation='relu', padding='same')(layers.UpSampling2D(size=(2, 2))(drop5))
    merge6 = layers.concatenate([drop4, up6], axis=3)
    conv6 = layers.Conv2D(512, 3, activation='relu', padding='same')(merge6)
    conv6 = layers.Conv2D(512, 3, activation='relu', padding='same')(conv6)
    
    up7 = layers.Conv2D(256, 2, activation='relu', padding='same')(layers.UpSampling2D(size=(2, 2))(conv6))
    merge7 = layers.concatenate([conv3, up7], axis=3)
    conv7 = layers.Conv2D(256, 3, activation='relu', padding='same')(merge7)
    conv7 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv7)
    
    up8 = layers.Conv2D(128, 2, activation='relu', padding='same')(layers.UpSampling2D(size=(2, 2))(conv7))
    merge8 = layers.concatenate([conv2, up8], axis=3)
    conv8 = layers.Conv2D(128, 3, activation='relu', padding='same')(merge8)
    conv8 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv8)
    
    up9 = layers.Conv2D(64, 2, activation='relu', padding='same')(layers.UpSampling2D(size=(2, 2))(conv8))
    merge9 = layers.concatenate([conv1, up9], axis=3)
    conv9 = layers.Conv2D(64, 3, activation='relu', padding='same')(merge9)
    conv9 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv9)
    
    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid')(conv9)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

def dice_coefficient(y_true, y_pred, smooth=1):
    """Dice coefficient metric"""
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    """Dice loss function"""
    return 1 - dice_coefficient(y_true, y_pred)

# Create model
model = create_unet_model(input_shape=(*CONFIG['image_size'], 3), num_classes=1)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss=dice_loss,
    metrics=['accuracy', dice_coefficient]
)

print(f"✅ Model created with {model.count_params():,} parameters")
model.summary()

## Step 6: Train Model

In [ ]:
# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(CONFIG['output_dir'], 'best_model.h5'),
        save_best_only=True,
        monitor='val_loss'
    ),
    keras.callbacks.EarlyStopping(
        patience=10,
        monitor='val_loss',
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        factor=0.5,
        patience=5,
        monitor='val_loss'
    )
]

# Train
print("🚀 Starting training...")
history = model.fit(
    train_images, train_masks,
    batch_size=CONFIG['batch_size'],
    epochs=CONFIG['epochs'],
    validation_data=(val_images, val_masks),
    callbacks=callbacks
)

print("✅ Training complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

# Dice Coefficient
axes[2].plot(history.history['dice_coefficient'], label='Training Dice')
axes[2].plot(history.history['val_dice_coefficient'], label='Validation Dice')
axes[2].set_title('Dice Coefficient')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Dice')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print("\n📊 Final Metrics:")
print(f"  Training Loss: {history.history['loss'][-1]:.4f}")
print(f"  Validation Loss: {history.history['val_loss'][-1]:.4f}")
print(f"  Training Dice: {history.history['dice_coefficient'][-1]:.4f}")
print(f"  Validation Dice: {history.history['val_dice_coefficient'][-1]:.4f}")

In [ ]:
# Visualize predictions
predictions = model.predict(val_images[:6])

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    axes[0, i].imshow(val_images[i])
    axes[0, i].set_title('Input Image')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(val_masks[i].squeeze(), cmap='gray')
    axes[1, i].set_title('Ground Truth')
    axes[1, i].axis('off')
    
    axes[2, i].imshow(predictions[i].squeeze(), cmap='gray')
    axes[2, i].set_title('Prediction')
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

## Step 7: Convert to CoreML

In [ ]:
# Save Keras model
model_path = os.path.join(CONFIG['output_dir'], f"{CONFIG['model_name']}.h5")
model.save(model_path)
print(f"✅ Keras model saved to {model_path}")

# Convert to CoreML
print("\n🔄 Converting to CoreML...")
coreml_model = ct.convert(
    model,
    inputs=[ct.ImageType(
        name="image",
        shape=(1, *CONFIG['image_size'], 3),
        scale=1/255.0,
        bias=[0, 0, 0]
    )],
    outputs=[ct.ImageType(name="segmentation_mask")],
    minimum_deployment_target=ct.target.iOS15
)

# Set metadata
coreml_model.author = "PediLens Team"
coreml_model.short_description = "Wound boundary segmentation model"
coreml_model.version = "1.0"

# Save CoreML model
coreml_path = os.path.join(CONFIG['output_dir'], f"{CONFIG['model_name']}.mlmodel")
coreml_model.save(coreml_path)

print(f"✅ CoreML model saved to {coreml_path}")
print(f"✅ Model size: {os.path.getsize(coreml_path) / (1024*1024):.2f} MB")

## Step 8: Download Model

In [ ]:
# Download CoreML model
from google.colab import files

print("📥 Downloading CoreML model...")
files.download(coreml_path)

print("\n✅ Download complete!")
print("\n📋 Next steps:")
print("1. Copy WoundSegmentation.mlmodel to your PediLens Xcode project")
print("2. Add it to PediLens/Resources/ folder")
print("3. Ensure it's added to the app target")
print("4. Build the project - Xcode will compile it to .mlmodelc")
print("5. The app will automatically use it for wound detection!")

## Optional: Save Training History

In [ ]:
# Save training history
history_path = os.path.join(CONFIG['output_dir'], 'training_history.json')
with open(history_path, 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    history_dict = {}
    for key, value in history.history.items():
        history_dict[key] = [float(v) for v in value]
    json.dump(history_dict, f, indent=2)

print(f"✅ Training history saved to {history_path}")

# Download history
files.download(history_path)